# Week 6 — **Reinforcement Fine‑Tuning (RFT)**: Product Pricer (Fixed)

This notebook **fixes the dataset format error** for Reinforcement Fine‑Tuning (RFT):
> **Invalid file format. Example 1, System messages are not supported in 'reinforcement' examples.**

**What changed?**  
RFT training **does not allow `system` messages** in the `messages` array.  
We now construct / convert examples so that **each example has only `role: "user"`** messages.

- If you already prepared **SFT JSONL** (with system/user/assistant), this notebook converts it to **RFT JSONL**.
- If you only have the original `train.pkl` / `test.pkl`, this notebook will build RFT JSONL directly **without** system messages.

> ✅ Works in **Google Colab**. A GPU (e.g., T4) is **not required** for OpenAI fine‑tuning jobs.

## 0) Install (optional)

In [1]:
# !pip -q install --upgrade openai wandb python-dotenv matplotlib numpy pandas tqdm requests

## 1) Environment & API keys

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
USE_WANDB = bool(os.getenv("WANDB_API_KEY"))

if USE_WANDB:
    import wandb
    wandb.login()

assert os.getenv("OPENAI_API_KEY") and os.getenv("OPENAI_API_KEY") != 'your-key-if-not-using-env', \
    "OPENAI_API_KEY is required. Please set it in a .env file or environment."

print("Environment ready.")

wandb: Currently logged in as: hafnium (hafnium49) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Environment ready.


## 2) Imports

In [3]:
import os, re, json, pickle, math, random
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from openai import OpenAI
client = OpenAI()

print("SDK ready.")

SDK ready.


## 3) Locate data (SFT JSONL **or** PKL)

In [4]:
SFT_TRAIN_JSONL = Path("fine_tune_train.jsonl")          # optional, if you already have SFT data
SFT_VAL_JSONL   = Path("fine_tune_validation.jsonl")     # optional, if you already have SFT data

DATA_DIR = Path(".")
TRAIN_PKL = DATA_DIR / "train.pkl"
TEST_PKL  = DATA_DIR / "test.pkl"

has_sft = SFT_TRAIN_JSONL.exists() and SFT_VAL_JSONL.exists()
has_pkl = TRAIN_PKL.exists() and TEST_PKL.exists()

print(f"SFT JSONL present: {has_sft}")
print(f"PKL present:       {has_pkl}")

if not has_sft and not has_pkl:
    print("⚠️ No SFT JSONL or PKL datasets found in current directory.")
    print("   - If your earlier uploads expired, please re-upload (e.g., train.pkl/test.pkl or fine_tune_*.jsonl).")

SFT JSONL present: True
PKL present:       True


### 3.1 Optional helpers (`items.Item`, `testing.Tester`)

In [5]:
Item = None
Tester = None
try:
    from items import Item
    from testing import Tester
    print("Imported items.Item and testing.Tester.")
except Exception as e:
    print("Using fallback Tester:", e)
    class _FallbackItem:
        def __init__(self, text, price):
            self.text = text
            self.price = float(price)
        def test_prompt(self):
            return f"How much does this cost?\n\n{self.text}\n\nPrice is $"
    class _FallbackTester:
        @staticmethod
        def test(fn, dataset, max_n=None, name="model"):
            maes = []
            n = len(dataset) if max_n is None else min(max_n, len(dataset))
            for i in range(n):
                guess, truth = fn(dataset[i]), float(dataset[i].price)
                err = abs(guess - truth); maes.append(err)
                color = "\x1b[92m" if err <= 50 else ("\x1b[93m" if err <= 150 else "\x1b[91m")
                print(f"{color}{i+1}: Guess: ${guess:.2f} Truth: ${truth:.2f} Error: ${err:.2f}\x1b[0m")
            print(f"\n{name} — MAE: ${np.mean(maes):.2f} | Median: ${np.median(maes):.2f}")
    Tester = _FallbackTester

Imported items.Item and testing.Tester.


## 4) Utilities

In [6]:
def get_price(s: str) -> float:
    s = (s or "").replace("$","").replace(",","")
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else 0.0

## 5) Convert **SFT JSONL** ➜ **RFT JSONL** (removes system messages)

In [7]:
def extract_user_content_from_messages(messages):
    '''Return concatenated user content from SFT messages (ignores system/assistant).'''
    parts = []
    for m in messages:
        if m.get("role") == "user":
            parts.append(m.get("content",""))
    return "\n".join(parts).strip()

def extract_reference_from_sft(messages) -> str:
    '''From SFT messages, take last assistant content and parse a price, fallback to empty.'''
    for m in reversed(messages):
        if m.get("role") == "assistant":
            price = get_price(m.get("content",""))
            if price:
                return f"{price:.2f}"
    return ""

def sft_jsonl_to_rft_jsonl(src_path: Path, dst_path: Path, max_rows: int = None):
    rows = 0
    with src_path.open("r", encoding="utf-8") as fin, dst_path.open("w", encoding="utf-8") as fout:
        for i, line in enumerate(fin):
            if line.strip() == "":
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue
            messages = obj.get("messages", [])
            # Build single user-only prompt (RFT restriction: no system messages)
            user_content = extract_user_content_from_messages(messages)
            instruction = "Estimate the price in USD. Output only a number (no $ or text)."
            user_prompt = instruction + "\n\n" + user_content.replace("\n\nPrice is $","")
            # Reference answer
            ref = obj.get("reference_answer") or extract_reference_from_sft(messages)
            if not ref:
                if "price" in obj:
                    ref = f"{float(obj['price']):.2f}"
                else:
                    continue
            new_obj = {
                "messages": [{"role":"user","content": user_prompt}],
                "reference_answer": ref
            }
            fout.write(json.dumps(new_obj, ensure_ascii=False) + "\n")
            rows += 1
            if max_rows and rows >= max_rows:
                break
    print(f"Converted {rows} rows -> {dst_path}")

## 6) Build **RFT JSONL** directly from PKL (no system messages)

In [8]:
def item_to_rft_row(item):
    instruction = "Estimate the price in USD. Output only a number (no $ or text)."
    user = item.test_prompt().replace(" to the nearest dollar","").replace("\n\nPrice is $","")
    return {
        "messages": [{"role":"user","content": instruction + "\n\n" + user}],
        "reference_answer": f"{float(item.price):.2f}"
    }

def write_rft_from_items(items, out_path: Path):
    n = 0
    with out_path.open("w", encoding="utf-8") as f:
        for it in items:
            f.write(json.dumps(item_to_rft_row(it), ensure_ascii=False) + "\n")
            n += 1
    print(f"Wrote {n} rows -> {out_path}")

## 7) Prepare RFT files

In [9]:
random.seed(42); np.random.seed(42)

OUT_DIR = Path("week6_rft_fixed"); OUT_DIR.mkdir(exist_ok=True, parents=True)
RFT_TRAIN = OUT_DIR / "rft_train.jsonl"
RFT_VAL   = OUT_DIR / "rft_val.jsonl"

N_TRAIN = 500
N_VAL   = 50

if has_sft:
    sft_jsonl_to_rft_jsonl(SFT_TRAIN_JSONL, RFT_TRAIN, max_rows=N_TRAIN)
    sft_jsonl_to_rft_jsonl(SFT_VAL_JSONL,   RFT_VAL,   max_rows=N_VAL)
elif has_pkl:
    with TRAIN_PKL.open("rb") as f: train = pickle.load(f)
    with TEST_PKL.open("rb") as f:  test  = pickle.load(f)
    if len(train)>0 and not hasattr(train[0], "test_prompt"):
        class _Item:
            def __init__(self, d):
                self.text  = d["text"]; self.price = float(d["price"])
            def test_prompt(self):
                return f"How much does this cost?\n\n{self.text}\n\nPrice is $"
        train = [ _Item(d) for d in train ]
        test  = [ _Item(d) for d in test ]
    idx = np.random.permutation(len(train))
    fit_sel = [train[i] for i in idx[:N_TRAIN]]
    val_sel = [train[i] for i in idx[N_TRAIN:N_TRAIN+N_VAL]]
    write_rft_from_items(fit_sel, RFT_TRAIN)
    write_rft_from_items(val_sel, RFT_VAL)
else:
    raise FileNotFoundError("No SFT JSONL or PKL available to prepare RFT data.")

Converted 200 rows -> week6_rft_fixed/rft_train.jsonl
Converted 50 rows -> week6_rft_fixed/rft_val.jsonl


## 8) Python grader (numeric accuracy reward)

In [10]:
import inspect, math, re, requests

def price_grader(sample, item):
    '''
    sample: {"output_text": "..."}  # model raw output
    item:   {"reference_answer": "123.45"}
    return: float in [0,1]
    '''
    def parse_num(s):
        if s is None: return None
        s = s.replace(",", "")
        m = re.search(r"[-+]?\d*\.?\d+", s)
        return float(m.group()) if m else None

    pred  = parse_num(sample.get("output_text", ""))
    truth = parse_num(item.get("reference_answer", ""))

    if pred is None or truth is None:
        return 0.0

    scale = max(25.0, 0.20 * max(1.0, truth))
    score = math.exp(-abs(pred - truth) / scale)

    if re.search(r"[^\d\.\-+]", (sample.get("output_text","").strip())):
        score *= 0.98

    return float(max(0.0, min(1.0, score)))

def build_python_grader_payload(grader_fn):
    src = inspect.getsource(grader_fn)
    if not src.strip().startswith("def grade("):
        src = src.replace(grader_fn.__name__, "grade", 1)
    return {"type":"python", "source": src}

GRADER_PAYLOAD = build_python_grader_payload(price_grader)
print("Grader ready.")

Grader ready.


### 8.1 Optional: validate grader with API

In [11]:
try:
    headers = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}
    resp = requests.post(
        "https://api.openai.com/v1/fine_tuning/alpha/graders/validate",
        json={"grader": GRADER_PAYLOAD},
        headers=headers, timeout=30
    )
    print("Validation:", resp.status_code, resp.text[:200])
except Exception as e:
    print("Grader validation skipped:", e)

Validation: 200 {
  "grader": {
    "type": "python",
    "source": "def grade(sample, item):\n    '''\n    sample: {\"output_text\": \"...\"}  # model raw output\n    item:   {\"reference_answer\": \"123.45\"}\n    


## 9) Upload files & launch **RFT**

In [12]:
BASE_MODEL = "o4-mini-2025-04-16"  # Reasoning model required for RFT
EPOCHS     = 3
REASONING  = "medium"              # none/low/medium/high

with RFT_TRAIN.open("rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")
with RFT_VAL.open("rb") as f:
    val_file = client.files.create(file=f, purpose="fine-tune")
print("Uploaded:", train_file.id, val_file.id)

integrations = []
if USE_WANDB:
    integrations = [{"type":"wandb", "wandb": {"project":"gpt-pricer-rft"}}]

job = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model=BASE_MODEL,
    seed=42,
    suffix=f"pricer-rft-fixed",
    method={
        "type":"reinforcement",
        "reinforcement":{
            "grader": GRADER_PAYLOAD,
            "hyperparameters":{
                "n_epochs": int(EPOCHS),
                "eval_interval": 5,
                "eval_samples": 3,
                "compute_multiplier": 1.0,
                "reasoning_effort": REASONING
            }
        }
    },
    integrations=integrations
)

print("RFT job:", job.id, "| status:", job.status)

Uploaded: file-6VR8ry1vi5hJsDyH5jP2vC file-Wf9xtXr236d7kn67PQgxrj
RFT job: ftjob-MjvUblvSlh2K7S2SEVhNJuhM | status: validating_files


## 10) Monitor job

In [13]:
def show_status(job_id, limit_events=10):
    j = client.fine_tuning.jobs.retrieve(job_id)
    print("Job:", j.id, "| status:", j.status, "| base:", j.model, "| fine_tuned:", j.fine_tuned_model)
    ev = client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=limit_events)
    for e in ev.data:
        print(f"- [{e.created_at}] {e.level}: {e.message}")
    return j

_ = show_status(job.id)

Job: ftjob-MjvUblvSlh2K7S2SEVhNJuhM | status: validating_files | base: o4-mini-2025-04-16 | fine_tuned: None
- [1759092520] info: Validating training file: file-6VR8ry1vi5hJsDyH5jP2vC and validation file: file-Wf9xtXr236d7kn67PQgxrj
- [1759092519] info: Created fine-tuning job: ftjob-MjvUblvSlh2K7S2SEVhNJuhM


## 11) Inference helpers (user-only prompts)

In [14]:
def rft_infer_messages(item):
    instruction = "Estimate the price in USD. Output only a number (no $ or text)."
    user = item.test_prompt().replace(" to the nearest dollar","").replace("\n\nPrice is $","")
    return [{"role":"user","content":instruction + "\n\n" + user}]

def gpt_rft_predict(item, model_name: str):
    rsp = client.chat.completions.create(
        model=model_name,
        messages=rft_infer_messages(item),
        temperature=0, max_tokens=8, seed=42
    )
    return get_price(rsp.choices[0].message.content)

## 12) Evaluate (after job succeeds)

In [15]:
SMOKE = 8  # cost-aware
try:
    _ = test
except NameError:
    print("Note: No PKL test set loaded. To evaluate, provide test.pkl or your own dataset.")
else:
    job_info = client.fine_tuning.jobs.retrieve(job.id)
    ft_model = job_info.fine_tuned_model
    if ft_model:
        print("Fine-tuned model:", ft_model)
        Tester.test(lambda it: gpt_rft_predict(it, ft_model), test, max_n=SMOKE, name="RFT (smoke)")
    else:
        print("Model not ready. Re-run this cell after job status == 'succeeded'.")

Note: No PKL test set loaded. To evaluate, provide test.pkl or your own dataset.


## 13) Notes & Tips

- ✅ **Fix implemented**: RFT examples now use **only `role: "user"` messages**. Any instruction that used to be in `system` is now **inlined into the user prompt**.
- If you previously prepared **SFT JSONL** (with system/user/assistant), use this notebook's **converter** to produce valid **RFT JSONL**.
- Start with small training sizes and epochs; iterate quickly and watch reward curves.
- If your earlier uploads (e.g., `train.pkl`/`test.pkl`) expired in Colab/Jupyter, **re-upload** them first.